# Get geo data

### Verejny dalkovy pristup k datum RUIAN

- [Adresy v CR (pozor, opravdu velky soubor)](https://vdp.cuzk.cz/vdp/ruian/vymennyformat?crKopie=on&casovyRozsah=U&svyAdresy=on&svyber=svyAdresy&search=)
- [Adresní místa RÚIAN ve formátu CSV](https://nahlizenidokn.cuzk.cz/StahniAdresniMistaRUIAN.aspx)
  - [Popis](https://vdp.cuzk.cz/vymenny_format/csv/hierarchie-prvku-ruian-popis.pdf)
- Encoding: Windows 1250


In [45]:
import pandas as pd

In [ ]:
fpath = 'Q:\\dev\\_projects\\github\\opendata_sandbox\\projects\\stredni_skoly\\vdp.cuzk.cz\\_data\\20240930_OB_ADR_csv\\CSV\\'
cities = list()

fname = '20240930_OB_500011_ADR.csv'
cities.append(pd.read_csv(fpath + fname, encoding='cp1250', sep=';'))
fname = '20240930_OB_500020_ADR.csv'
cities.append(pd.read_csv(fpath + fname, encoding='cp1250', sep=';'))
fname = '20240930_OB_500046_ADR.csv'
cities.append(pd.read_csv(fpath + fname, encoding='cp1250', sep=';'))
df_3_cities = pd.concat(cities, ignore_index=True)
df_3_cities.tail(5)


In [ ]:
from pathlib import Path
fpath = 'Q:\\dev\\_projects\\github\\opendata_sandbox\\projects\\stredni_skoly\\vdp.cuzk.cz\\_data\\20240930_OB_ADR_csv\\CSV\\'

cities = list()

# fname = '20240930_OB_500011_ADR.csv'


files = Path(fpath).glob('*.csv')
for file in files:
    cities.append(pd.read_csv(file, encoding='cp1250', sep=';'))

print(f'Processed files: {len(cities)}')

In [ ]:
df_all_cities = pd.concat(cities, ignore_index=True)
df_all_cities.tail(5)

In [ ]:
df_all_cities.info()

In [50]:
# store it back to csv
csv_path = 'Q:\\dev\\_projects\\github\\opendata_sandbox\\projects\\stredni_skoly\\vdp.cuzk.cz\\_data\\out\\'
csv_fname = 'all_locations.csv'
df_all_cities.to_csv(csv_path + csv_fname, index_label='ROWID_COORD_TMP')

## Transform coordinates

### Let's start with one file only

In [ ]:
from pyproj import Transformer

transformer = Transformer.from_crs("EPSG:5514", "EPSG:4326" )
a, b = 772792.67, 1055433.02
    # Melo by mi to prevest na 49.9394242N, 14.0310217E
    # bacha, musi se to davat zaporne a obracene, pokud se to bere z dat z VFR
    # viz https://cuzk.gov.cz/ruian/Poskytovani-udaju-ISUI-RUIAN-VDP/Vymenny-format-RUIAN-(VFR)/FAQ-casto-kladene-otazky-k-VFR/10-Jaky-format-souradnic-je-ve-VFR-vyuzivan.asp


lat, long = transformer.transform(-a, -b)
lat, long


In [ ]:
fname = '20240930_OB_533203_ADR.csv'
city = pd.read_csv(fpath + fname, encoding='cp1250', sep=';')
city['lat'] = city.apply(lambda x: transformer.transform(-x['Souřadnice Y'], -x['Souřadnice X'])[0], axis = 1)
# city['lat_long']
city['long'] = city.apply(lambda x: transformer.transform(-x['Souřadnice Y'], -x['Souřadnice X'])[1], axis = 1)
city.head(5)


### Tak a jdu prekonvertit vsechna data



In [ ]:
df_all_cities['lat'] = df_all_cities.apply(lambda x: transformer.transform(-x['Souřadnice Y'], -x['Souřadnice X'])[0], axis = 1)
df_all_cities['long'] = df_all_cities.apply(lambda x: transformer.transform(-x['Souřadnice Y'], -x['Souřadnice X'])[1], axis = 1)
df_all_cities.head(5)


In [ ]:
df_all_cities.info()

In [55]:
csv_fname = 'all_locations_with_lat_long.csv'
df_all_cities.to_csv(csv_path + csv_fname, index_label='ROWID_COORD_TMP')